# Environment Check: Smoke Test

Dieses Notebook prüft, ob alle Libraries installiert sind und die Snowflake-Verbindung (.env) funktioniert.
**Nach jedem Setup ausführen. Alle Zellen müssen ohne Fehler durchlaufen.**

Kernel: `.venv` (Setup mit uv) oder `python.exe` im WinPython-Ordner (Setup mit WinPython, siehe README).
Fehlt ein Paket, bei WinPython nichts selbst installieren, sondern beim WinPython-Verantwortlichen anfragen.
Die Snowflake-Zelle darf fehlschlagen, solange noch nicht mit Snowflake gearbeitet wird.

In [ ]:
# Python & System
import sys
print(f"Python: {sys.version}")
print(f"Platform: {sys.platform}")

In [ ]:
# Core Libraries
import pandas as pd
import numpy as np
import sklearn
import matplotlib
print(f"pandas:       {pd.__version__}")
print(f"numpy:        {np.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"matplotlib:   {matplotlib.__version__}")
print("Core Libraries OK")

In [ ]:
# ML Libraries
import lightgbm as lgb
import xgboost as xgb
import statsmodels
import optuna
import shap
print(f"LightGBM:     {lgb.__version__}")
print(f"XGBoost:      {xgb.__version__}")
print(f"statsmodels:  {statsmodels.__version__}")
print(f"Optuna:       {optuna.__version__}")
print(f"SHAP:         {shap.__version__}")
print("ML Libraries OK")

In [ ]:
# Time Series / Deep Learning
# Missing packages: uv -> "uv sync --extra cpu|gpu", WinPython -> ask the WinPython maintainer (README)
import warnings

try:
    from neuralforecast import NeuralForecast
    from neuralforecast.models import NHITS
    print("NeuralForecast (N-HiTS) OK")
except ImportError:
    print("NeuralForecast nicht installiert: uv sync ausführen (WinPython: Paket beim WinPython-Verantwortlichen anfragen)")

try:
    import torch
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA:    {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU:     {torch.cuda.get_device_name(0)}")
        # A detected GPU is not enough: the PyTorch build must ship kernels for its compute capability
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                (torch.ones(3, device="cuda") * 2).sum().item()
            print("GPU-Test: OK, PyTorch rechnet auf der GPU")
        except Exception as e:
            major, minor = torch.cuda.get_device_capability(0)
            print(f"GPU-Test: FEHLGESCHLAGEN ({type(e).__name__})")
            print(f"  Dieses PyTorch ({torch.__version__}) unterstützt die GPU nicht (Compute Capability {major}.{minor}).")
            print("  Neuronale Modelle laufen vorerst nur auf der CPU. Lösung: PyTorch für CUDA 13.0 (cu130),")
            print("  bei uv: uv sync --extra gpu, bei WinPython: beim WinPython-Verantwortlichen anfragen.")
except ImportError:
    print("PyTorch nicht installiert: uv sync --extra cpu oder --extra gpu (WinPython: beim WinPython-Verantwortlichen anfragen)")

In [ ]:
# Snowflake-Verbindung
try:
    from src.data_loader import get_session
    session = get_session()
    result = session.sql(
        "SELECT CURRENT_TIMESTAMP() AS ts, CURRENT_ROLE() AS role, CURRENT_WAREHOUSE() AS wh"
    ).to_pandas()
    print(f"Timestamp: {result['TS'].iloc[0]}")
    print(f"Rolle:     {result['ROLE'].iloc[0]}")
    print(f"Warehouse: {result['WH'].iloc[0]}")
    print("Snowflake-Verbindung OK")
except Exception as e:
    print(f"Snowflake-Verbindung fehlgeschlagen: {e}")

In [ ]:
# Config laden
from src.config import load_config, RANDOM_STATE
cfg = load_config()
print(f"Projekt:      {cfg['project']['name']}")
print(f"Version:      {cfg['project']['version']}")
print(f"Datenbank:    {cfg['snowflake']['database']}")
print(f"Random State: {RANDOM_STATE}")
print("Config OK")

---
Wenn alle Zellen durchlaufen: Umgebung bereit. Weiter mit `01_data_exploration.ipynb`